# Comparative Sentiment Analysis: Indonesian E-Commerce Reviews (Scopus Pipeline)
## Notebook 05: Comparative Analysis, Ablation Study, & Qualitative Linguistic Error Analysis

```text
Author: Muhammad Farrel Haidar
Project: AI PR & KOL Specialist DSS
Date: 2026-10-14
Study: Comparative Sentiment Analysis (IndoBERT vs Tree-Based Models)
Scope: 3 Models Benchmark, Ablation Study (Table 4), Statistical Significance (McNemar), Error Analysis
Target Artifacts: Table 2-4, Figure 2, 4, 5 (300 DPI)
```

---
### 1. Metodologi Analisis Komparatif Komprehensif (Opsi B)
Notebook ini menyajikan evaluasi mendalam terhadap seluruh hipotesis riset:
1. **Komparasi Kinerja Global (Table 2)**: Evaluasi akurasi, Macro-F1, Weighted-F1, ROC-AUC, dan waktu inferensi.
2. **Kinerja Per-Kelas (Table 3 & Figure 5)**: Analisis resolusi polaritas pada kelas sulit (Netral) pasca-penerapan weighted loss.
3. **Studi Ablasi (Table 4)**: Menguji kontribusi modular komponen normalisasi slang, disambiguasi rating 3, extended length 256, dan class-weighted loss.
4. **Reliabilitas Ground-Truth**: Menampilkan hasil kesepakatan inter-annotator Cohen's Kappa ($\kappa = 0.8939$).
5. **Signifikansi Statistik (McNemar's Test)**: Uji signifikansi non-parametrik antara IndoBERT dan XGBoost.
6. **Analisis Galat Kualitatif (Error Analysis)**: Membedah 20 kasus linguistik kompleks (negasi bertingkat, sarkasme, konjungsi kontrastif).

In [ ]:
# Author: Muhammad Farrel Haidar
# Project: AI PR & KOL Specialist DSS
# Date: 2026-10-14

import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

sns.set_theme(style="whitegrid", font="DejaVu Sans")
plt.rcParams["figure.figsize"] = (10, 6)
print("Environment initialized successfully.")

### 2. Table 2: Model Performance Comparison (Test Set)

In [ ]:
t2_path = "../paper_assets/table2_model_comparison.csv" if os.path.exists("../paper_assets") else "paper_assets/table2_model_comparison.csv"
t2 = pd.read_csv(t2_path)
display(t2)

### 3. Table 3: Detailed Per-Class Metrics

In [ ]:
t3_path = "../paper_assets/table3_per_class_metrics.csv" if os.path.exists("../paper_assets") else "paper_assets/table3_per_class_metrics.csv"
t3 = pd.read_csv(t3_path)
display(t3)

### 3b. Table 4: Ablation Study (Pengaruh Slang, Disambiguasi, Panjang Konteks, & Loss Berbobot)
Studi ablasi mengisolasi dampak dari setiap intervensi teknik terhadap performa IndoBERT:

In [ ]:
t4_path = "../paper_assets/table4_ablation_study.csv" if os.path.exists("../paper_assets/table4_ablation_study.csv") else "paper_assets/table4_ablation_study.csv"
if os.path.exists(t4_path):
    t4 = pd.read_csv(t4_path)
    display(t4)
else:
    print("Table 4 not found.")

### 3c. Inter-Annotator Agreement (Cohen's Kappa Reliability)
Tingkat reliabilitas anotasi ground truth dievaluasi menggunakan koefisien Cohen's Kappa pada 500 sampel acak terstratifikasi:

In [ ]:
kappa_path = "../paper_assets/cohen_kappa_validation.json" if os.path.exists("../paper_assets/cohen_kappa_validation.json") else "paper_assets/cohen_kappa_validation.json"
if os.path.exists(kappa_path):
    with open(kappa_path, "r", encoding="utf-8") as f:
        k_data = json.load(f)
    print(json.dumps(k_data, indent=2))

### 4. Visualisasi Figure 5: Grouped Bar Chart F1-Score Per Kelas

In [ ]:
plot_df = t3.copy()
plot_df["F1_Numeric"] = plot_df["F1-Score"].str.rstrip("%").astype(float)

plt.figure(figsize=(9, 5.5), dpi=300)
palette = ["#708090", "#e67e22", "#2e6f40"]
ax = sns.barplot(data=plot_df, x="Sentiment_Class", y="F1_Numeric", hue="Model", palette=palette)
plt.title("Figure 5: Per-Class F1-Score Comparison Across Model Architectures", fontsize=12, pad=15, fontweight="bold")
plt.xlabel("Sentiment Class", fontsize=11, labelpad=8)
plt.ylabel("F1-Score (%)", fontsize=11, labelpad=8)
plt.ylim(0, 100)
plt.legend(title="Model Architecture", loc="upper right")
for p in ax.patches:
    h = p.get_height()
    if h > 0:
        ax.annotate(f"{h:.1f}%", (p.get_x() + p.get_width() / 2.0, h), ha="center", va="bottom", fontsize=9, xytext=(0, 3), textcoords="offset points")
plt.tight_layout()
plt.show()

### 5. Visualisasi Figure 2: Word Cloud Per Kelas Sentimen

In [ ]:
from PIL import Image
fig2_path = "../paper_assets/figure2_wordcloud_per_class.png" if os.path.exists("../paper_assets") else "paper_assets/figure2_wordcloud_per_class.png"
if os.path.exists(fig2_path):
    img = Image.open(fig2_path)
    plt.figure(figsize=(15, 5), dpi=200)
    plt.imshow(img)
    plt.axis("off")
    plt.show()

### 6. Visualisasi Figure 4: Side-by-Side Confusion Matrix (3 Model)

In [ ]:
fig4_path = "../paper_assets/figure4_confusion_matrix_comparison.png" if os.path.exists("../paper_assets") else "paper_assets/figure4_confusion_matrix_comparison.png"
if os.path.exists(fig4_path):
    img = Image.open(fig4_path)
    plt.figure(figsize=(16, 5), dpi=200)
    plt.imshow(img)
    plt.axis("off")
    plt.show()

### 7. Uji Signifikansi Statistik McNemar's Test

In [ ]:
mcnemar_path = "../data/results/mcnemar_test_result.json" if os.path.exists("../data/results") else "data/results/mcnemar_test_result.json"
if os.path.exists(mcnemar_path):
    with open(mcnemar_path, "r", encoding="utf-8") as f:
        mcnemar_data = json.load(f)
    print(json.dumps(mcnemar_data, indent=2, ensure_ascii=False))

### 8. Qualitative Error Analysis (20 Sampel Fenomena Linguistik)

In [ ]:
err_path = "../paper_assets/error_analysis_samples.csv" if os.path.exists("../paper_assets") else "paper_assets/error_analysis_samples.csv"
err_df = pd.read_csv(err_path)
print(f"Total Error Analysis Samples: {len(err_df)}")
display(err_df[["Sample_ID", "Application", "True_Sentiment", "IndoBERT_Prediction", "RandomForest_Prediction", "XGBoost_Prediction", "Linguistic_Phenomenon", "Original_Content"]].head(10))